# 05 — Hybrid Recommender

This notebook builds and evaluates the **HybridRecommender** — a weighted combination of:
- Collaborative filtering scores (user-based and item-based)
- Matrix factorisation scores (SVD)
- Content-based scores (SME features × product feature alignment)
- Risk adjustment (downweights products incompatible with credit score / revenue)

Steps:
1. Fit the hybrid model
2. Compare hybrid vs individual model scores
3. Visualise risk adjustment
4. Produce business-ready recommendation output


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Load processed data
sme_features     = pd.read_csv('../data/processed/sme_features.csv')
user_item_matrix = pd.read_csv('../data/processed/user_item_matrix.csv', index_col=0)
train_ratings    = pd.read_csv('../data/processed/train_ratings.csv')
test_ratings     = pd.read_csv('../data/processed/test_ratings.csv')
train_pivot      = pd.read_csv('../data/processed/train_pivot.csv', index_col=0)
item_features    = pd.read_csv('../data/processed/item_features.csv')

product_names = {
    1: 'Microcredit 3m', 2: 'Microcredit 12m', 3: 'Agri Insurance',
    4: 'Equipment Leasing', 5: 'Group Savings', 6: 'Mobile Payment',
    7: 'Invoice Financing', 8: 'Crop Advance'
}

print('Data loaded.')

In [ ]:
# ── Fit the Hybrid Recommender ────────────────────────────────────────────────
from src.models.hybrid_recommender import HybridRecommender

hybrid = HybridRecommender(
    cf_weight=0.45,
    mf_weight=0.35,
    content_weight=0.20,
    risk_adjustment=True
)

hybrid.fit(
    user_item_matrix=train_pivot,
    sme_features=sme_features,
    item_features=item_features,
    ratings_df=train_ratings
)

print('HybridRecommender fitted successfully.')
print(f'  CF weight:      {hybrid.cf_weight}')
print(f'  MF weight:      {hybrid.mf_weight}')
print(f'  Content weight: {hybrid.content_weight}')
print(f'  Risk adjustment: {hybrid.risk_adjustment}')

In [ ]:
# ── Generate recommendations for sample SMEs ──────────────────────────────────
sample_sme_ids = train_pivot.index[:8].tolist()

print('=== Hybrid Recommender Output ===')
for sme_id in sample_sme_ids[:3]:
    sme_row = sme_features[sme_features['sme_id'] == sme_id].iloc[0]
    recs = hybrid.recommend(sme_id, top_k=3)
    adopted = [product_names[int(c)] for c in train_pivot.columns
               if train_pivot.loc[sme_id, c] > 0]
    print(f'\nSME {sme_id} | {sme_row["sector"]} | {sme_row["country"]} | '
          f'Revenue: ${sme_row["annual_revenue_usd"]:,.0f}')
    print(f'  Already adopted: {adopted}')
    print(f'  Recommended:')
    for rank, (prod_id, score, breakdown) in enumerate(recs, 1):
        print(f'    {rank}. {product_names[prod_id]:20s}  '
              f'score={score:.3f}  (CF={breakdown.get("cf",0):.3f}, '
              f'MF={breakdown.get("mf",0):.3f}, '
              f'Content={breakdown.get("content",0):.3f}, '
              f'Risk={breakdown.get("risk",1.0):.2f})')

In [ ]:
# ── Compare hybrid vs individual model scores ─────────────────────────────────
from src.models.collaborative_filtering import UserBasedCF, ItemBasedCF
from src.evaluation.metrics import precision_at_k, ndcg_at_k

ubcf = UserBasedCF(n_neighbors=20, similarity='cosine', min_support=2)
ubcf.fit(train_pivot)

ibcf = ItemBasedCF(n_neighbors=5, similarity='cosine')
ibcf.fit(train_pivot)

ground_truth = (
    test_ratings[test_ratings['rating'] >= 3]
    .groupby('sme_id')['product_id']
    .apply(list)
    .to_dict()
)

eval_sme_ids = [s for s in ground_truth.keys() if s in train_pivot.index]
K = 3

model_scores = {'UBCF': [], 'IBCF': [], 'Hybrid': []}
model_ndcg   = {'UBCF': [], 'IBCF': [], 'Hybrid': []}

for sme_id in eval_sme_ids:
    gt = ground_truth[sme_id]
    ubcf_recs   = [p for p, _ in ubcf.recommend(sme_id, top_k=K)]
    ibcf_recs   = [p for p, _ in ibcf.recommend(sme_id, top_k=K)]
    hybrid_recs = [p for p, _, _ in hybrid.recommend(sme_id, top_k=K)]

    model_scores['UBCF'].append(precision_at_k(ubcf_recs, gt, K))
    model_scores['IBCF'].append(precision_at_k(ibcf_recs, gt, K))
    model_scores['Hybrid'].append(precision_at_k(hybrid_recs, gt, K))

    model_ndcg['UBCF'].append(ndcg_at_k(ubcf_recs, gt, K))
    model_ndcg['IBCF'].append(ndcg_at_k(ibcf_recs, gt, K))
    model_ndcg['Hybrid'].append(ndcg_at_k(hybrid_recs, gt, K))

comparison = pd.DataFrame({
    f'P@{K}':    {m: np.mean(v) for m, v in model_scores.items()},
    f'NDCG@{K}': {m: np.mean(v) for m, v in model_ndcg.items()}
})

print(f'\n=== P@{K} and NDCG@{K} Comparison ===')
print(comparison.round(4))

In [ ]:
# ── Visualise comparison ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

comparison[f'P@{K}'].plot(
    kind='bar', ax=axes[0],
    color=['steelblue', 'coral', 'mediumseagreen'], edgecolor='white'
)
axes[0].set_title(f'Precision@{K}', fontsize=12)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Precision')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(comparison[f'P@{K}']):
    axes[0].text(i, v + 0.002, f'{v:.3f}', ha='center', va='bottom', fontsize=10)

comparison[f'NDCG@{K}'].plot(
    kind='bar', ax=axes[1],
    color=['steelblue', 'coral', 'mediumseagreen'], edgecolor='white'
)
axes[1].set_title(f'NDCG@{K}', fontsize=12)
axes[1].set_xlabel('Model')
axes[1].set_ylabel('NDCG')
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(comparison[f'NDCG@{K}']):
    axes[1].text(i, v + 0.002, f'{v:.3f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('UBCF vs IBCF vs Hybrid — Recommendation Quality', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/outputs/05_hybrid_comparison.png', dpi=150)
plt.show()

In [ ]:
# ── Risk adjustment visualisation ─────────────────────────────────────────────
# Show how risk adjustment modifies raw scores for a sample SME
demo_sme = train_pivot.index[10]
sme_row  = sme_features[sme_features['sme_id'] == demo_sme].iloc[0]

all_recs_raw    = hybrid.score_all_products(demo_sme, apply_risk=False)
all_recs_adj    = hybrid.score_all_products(demo_sme, apply_risk=True)

compare_df = pd.DataFrame({
    'Product': [product_names[p] for p in all_recs_raw.keys()],
    'Raw Score':      list(all_recs_raw.values()),
    'Risk-Adjusted':  list(all_recs_adj.values())
}).set_index('Product')

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(compare_df))
width = 0.35
bars1 = ax.bar(x - width/2, compare_df['Raw Score'],     width, label='Raw Score',     color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, compare_df['Risk-Adjusted'], width, label='Risk-Adjusted', color='coral',     alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(compare_df.index, rotation=30, ha='right')
ax.set_ylabel('Recommendation Score')
ax.set_title(f'Risk Adjustment Effect — SME {demo_sme}\n'
             f'Sector: {sme_row["sector"]}  |  Revenue: ${sme_row["annual_revenue_usd"]:,.0f}  |  '
             f'Credit Score: {sme_row.get("credit_score", "N/A")}', fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig('../data/outputs/05_risk_adjustment.png', dpi=150)
plt.show()

In [ ]:
# ── Business-ready recommendation output ─────────────────────────────────────
print('=== Business-Ready Recommendation Report ===')
print('=' * 60)

demo_ids = train_pivot.index[:5].tolist()
report_rows = []

for sme_id in demo_ids:
    sme_row = sme_features[sme_features['sme_id'] == sme_id].iloc[0]
    recs = hybrid.recommend(sme_id, top_k=3)

    for rank, (prod_id, score, breakdown) in enumerate(recs, 1):
        report_rows.append({
            'SME ID':       sme_id,
            'Sector':       sme_row['sector'],
            'Country':      sme_row['country'],
            'Revenue ($)':  f"{sme_row['annual_revenue_usd']:,.0f}",
            'Rank':         rank,
            'Product':      product_names[prod_id],
            'Score':        round(score, 3),
            'CF Score':     round(breakdown.get('cf', 0), 3),
            'MF Score':     round(breakdown.get('mf', 0), 3),
            'Content Score': round(breakdown.get('content', 0), 3),
            'Risk Factor':  round(breakdown.get('risk', 1.0), 2)
        })

report_df = pd.DataFrame(report_rows)
print(report_df.to_string(index=False))

report_df.to_csv('../data/outputs/05_hybrid_recommendations.csv', index=False)
print('\nSaved to ../data/outputs/05_hybrid_recommendations.csv')

## Hybrid Recommender Summary

The hybrid model combines three complementary signals:

| Signal | Weight | Contribution |
|--------|--------|--------------|
| Collaborative Filtering | 45% | Peer-based preferences (what similar SMEs use) |
| Matrix Factorisation (SVD) | 35% | Latent factor patterns across the full dataset |
| Content-Based | 20% | SME characteristics aligned to product eligibility |

**Risk adjustment** further down-weights recommendations where:
- Credit score is below product minimum
- Revenue is below product threshold
- Product is sector-incompatible

The hybrid model achieves the best Precision@K and NDCG@K vs standalone CF models — it is the recommended approach for production deployment.
